# memopro E070: Windows PC (CPU) 한계 시험과 비교 (docs/research/0364)

학교 PC 같은 **Windows + CPU만 있는 기계**에서 memopro를 1시간 안에 재고, 같은 작업을 Transformers·llama.cpp·DuckDB·pandas와 비교한다. Colab의 **로컬 런타임**으로 이 PC의 Python에서 돈다(GPU는 쓰지 않음: Intel UHD 그래픽은 PyTorch가 쓰지 못한다).

| 갈래 | 무엇을 보나 |
|---|---|
| 생성 | Qwen2.5-1.5B(가중치 3.1GB)를 프로세스 상한 2GiB·1.5GiB에서 `memopro.generate`: 글이 그냥 올린 모델과 같은가(W1), 최대 메모리가 상한 안인가(W3). llama.cpp 4비트·16비트와 속도 비교(보고) |
| 학습 | Qwen2.5-0.5B 16비트 LoRA를 상한 2·3GiB에서 `memopro.finetune`: 손실이 상한과 무관하게 같은가(W2) |
| 데이터 | 4GB CSV를 1.5GiB 상한에서 `memopro.sort`/`join`: 결과가 DuckDB와 같은가(W4), 디스크에 쓰지 않는가(W5). pandas는 메모리에 다 올림(보고) |
| 압축 | 런타임 무손실 압축의 비율·속도(보고) |

**준비 (처음 한 번)**
1. 이 PC에 Python 3.11~3.13과 Jupyter를 설치한다(`pip install notebook`). 디스크 여유 약 **20GB**(모델 6GB, CSV 4GB, DuckDB 임시 파일 포함).
2. 명령 프롬프트에서 실행한다:
   `jupyter notebook --NotebookApp.allow_origin="https://colab.research.google.com" --port=8888 --NotebookApp.port_retries=0`
3. Colab에서 이 노트북을 열고 **연결 → 로컬 런타임에 연결**을 고른 뒤, 2번이 출력한 주소(토큰 포함)를 붙여 넣는다.
4. 셀을 실행한다. 필요한 패키지(memopro, torch CPU, DuckDB 등)는 셀이 설치한다. 다른 프로그램은 닫아 두는 것이 좋다.

**결과**: `%USERPROFILE%\memopro_e070\runs\<시각>\summary.md`와 같은 이름의 `.zip`(마지막 줄에 경로가 나온다). zip을 보내 주면 된다.


In [ ]:
# ==== SETTINGS (edit here) ===========================================================
# E070 (pre-registered in docs/research/0364): memopro on a Windows PC (CPU only) through a
# Colab local runtime, within an hour, next to Transformers, llama.cpp, DuckDB and pandas.
TIME_LIMIT_S = 55 * 60  # cases that would end after this are skipped (setup counts too)
PIP_SPEC = "memopro[llm]>=0.2.0b4"
GEN_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"  # 3.1 GB of bf16 weights
GEN_CEILINGS = [2 << 30, 1536 << 20]  # whole-process ceilings below the weights
NEW_TOKENS = 32
LORA_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"  # 1.0 GB
LORA_CEILINGS = [2 << 30, 3 << 30]
LORA_STEPS, LORA_SEQ = 4, 129
GGUF_REPO = "Qwen/Qwen2.5-1.5B-Instruct-GGUF"
GGUF_KINDS = ["q4_k_m", "fp16"]  # llama.cpp: 4-bit (lossy) and 16-bit
DATA_GIB = 4  # fact.csv size (needs about 3x this free on the disk, DuckDB's temp included)
DATA_DIMS = 2_000_000
DATA_CEILING = 1536 << 20  # memopro's ceiling and DuckDB's memory limit
PANDAS = True  # pandas reads the whole file (about 3x its size in memory)
BASE = None  # results and caches; None = %USERPROFILE%\memopro_e070
# ==== END OF SETTINGS ================================================================
import json
import os
import shutil
import subprocess
import sys
import time

T_START = time.time()
WORKERS = {'worker_windows.py': '"""E070 worker (docs/research/0364): one case in a fresh process on a Windows (or any) CPU machine.\n\n    python worker_windows.py OUT.json KIND [args...]\n\nEvery result records the process\'s peak working set (psutil ``peak_wset`` on Windows, peak RSS\nelsewhere) and the bytes it wrote (psutil I/O counters). Outputs of sorts and joins are hashed as\nthey stream (SHA-256 for sorts, an order-free row hash for joins), never written.\n"""\n\nimport hashlib\nimport json\nimport os\nimport sys\nimport time\nimport traceback\n\nOUT, KIND, ARGS = sys.argv[1], sys.argv[2], sys.argv[3:]\nRESULT = {"kind": KIND, "args": ARGS, "status": None}\nPROMPTS = [\n    "Explain how a hash table handles collisions.",\n    "Write a Python function that checks whether a string is a palindrome, with a short docstring.",\n]\nTEXT = (\n    "Memory is the scarcest resource on small machines. A program that needs more than the "\n    "machine has either fails, swaps to the disk, or is changed to need less. Lossless methods "\n    "keep the results exactly as they were: they read data again from its file, compress what "\n    "waits, and recompute what can be recomputed. "\n)\n\n\ndef proc_stats():\n    """Peak memory and bytes written by this process (Windows: peak working set and I/O\n    counters; macOS has no I/O counters in psutil)."""\n    import psutil\n\n    p = psutil.Process()\n    peak = getattr(p.memory_info(), "peak_wset", None) or _maxrss()\n    io = p.io_counters() if hasattr(p, "io_counters") else None\n    return {\n        "peak_bytes": peak,\n        "written_bytes": io.write_bytes if io else None,\n        "read_bytes": io.read_bytes if io else None,\n    }\n\n\ndef _maxrss():\n    import resource\n\n    r = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss\n    return r if sys.platform == "darwin" else r * 1024\n\n\nclass RowHash:\n    """Order-free hash of the rows it is fed, header (first line) left out: the sum of 64-bit\n    row hashes mod 2**64, and the row count. Rows never hold line breaks here."""\n\n    def __init__(self):\n        self.carry, self.total, self.rows, self.header = b"", 0, 0, None\n\n    def update(self, data):\n        data = self.carry + bytes(data)\n        cut = data.rfind(b"\\n") + 1\n        self.carry = data[cut:]\n        lines = data[:cut].split(b"\\n")[:-1]\n        if self.header is None and lines:\n            self.header, lines = lines[0], lines[1:]\n        for line in lines:\n            line = line.rstrip(b"\\r")\n            self.total = (\n                self.total + int.from_bytes(hashlib.blake2b(line, digest_size=8).digest(), "little")\n            ) % (1 << 64)\n        self.rows += len(lines)\n\n    def value(self):\n        return f"{self.total:016x}:{self.rows}"\n\n\ndef tok_and_model_dir(path):\n    import transformers\n\n    return transformers.AutoTokenizer.from_pretrained(path)\n\n\ndef chat_ids(tok, prompt):\n    ids = tok.apply_chat_template(\n        [{"role": "user", "content": prompt}], add_generation_prompt=True, return_tensors="pt"\n    )\n    return ids["input_ids"] if hasattr(ids, "keys") else ids\n\n\ndef gen_plain(path, n):\n    import torch\n    import transformers\n\n    tok = tok_and_model_dir(path)\n    t = time.time()\n    m = transformers.AutoModelForCausalLM.from_pretrained(path, dtype=torch.bfloat16)\n    RESULT["load_s"] = time.time() - t\n    texts, secs = [], []\n    for p in PROMPTS:\n        ids = chat_ids(tok, p)\n        t = time.time()\n        with torch.no_grad():\n            out = m.generate(\n                input_ids=ids,\n                do_sample=False,\n                max_new_tokens=n,\n                pad_token_id=tok.pad_token_id or tok.eos_token_id,\n            )\n        secs.append(time.time() - t)\n        texts.append(tok.decode(out[0, ids.shape[1] :], skip_special_tokens=True))\n    RESULT.update(texts=texts, seconds=secs, s_per_token=sum(secs) / _tokens(tok, texts))\n\n\ndef _tokens(tok, texts):\n    return max(1, sum(len(tok(t, add_special_tokens=False).input_ids) for t in texts))\n\n\ndef gen_mp(path, n, ceiling):\n    import memopro\n\n    tok = tok_and_model_dir(path)\n    texts, secs = [], []\n    for p in PROMPTS:\n        t = time.time()\n        texts.append(\n            memopro.generate(\n                path, p, tokenizer=tok, budget=int(ceiling), device="cpu", max_new_tokens=n\n            )\n        )\n        secs.append(time.time() - t)\n    RESULT.update(\n        texts=texts, seconds=secs, s_per_token=sum(secs) / _tokens(tok, texts), ceiling=int(ceiling)\n    )\n\n\ndef gen_llama(gguf, n):\n    from llama_cpp import Llama\n\n    t = time.time()\n    m = Llama(model_path=gguf, n_ctx=1024, n_threads=os.cpu_count(), verbose=False, seed=0)\n    RESULT["load_s"] = time.time() - t\n    texts, secs = [], []\n    for p in PROMPTS:\n        t = time.time()\n        r = m.create_chat_completion(\n            messages=[{"role": "user", "content": p}], max_tokens=n, temperature=0.0, top_k=1\n        )\n        secs.append(time.time() - t)\n        texts.append(r["choices"][0]["message"]["content"])\n        RESULT.setdefault("tokens", []).append(r["usage"]["completion_tokens"])\n    RESULT.update(\n        texts=texts,\n        seconds=secs,\n        s_per_token=sum(secs) / max(1, sum(RESULT["tokens"])),\n        gguf=os.path.basename(gguf),\n    )\n\n\ndef lora_texts(steps, seq):\n    return [f"{i}. " + TEXT * (1 + seq // 60) for i in range(steps)]\n\n\ndef lora_plain(path, steps, seq):\n    import peft\n    import torch\n    import transformers\n\n    torch.manual_seed(0)\n    tok = tok_and_model_dir(path)\n    m = transformers.AutoModelForCausalLM.from_pretrained(path, dtype=torch.bfloat16)\n    m = peft.get_peft_model(\n        m,\n        peft.LoraConfig(\n            r=8, lora_alpha=16, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]\n        ),\n    )\n    for p in m.parameters():\n        if p.requires_grad:\n            p.data = p.data.float()\n    ids = tok((tok.eos_token or "").join(lora_texts(steps, seq)), return_tensors="pt").input_ids[0]\n    opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad], lr=2e-4)\n    losses, secs = [], []\n    for i in range(steps):\n        x = ids[i * seq : (i + 1) * seq][None]\n        t = time.time()\n        loss = m(input_ids=x, labels=x).loss\n        loss.backward()\n        opt.step()\n        opt.zero_grad(set_to_none=True)\n        secs.append(time.time() - t)\n        losses.append(float(loss))\n    RESULT.update(losses=losses, step_s=secs)\n\n\ndef lora_mp(path, steps, seq, ceiling):\n    import memopro\n\n    tok = tok_and_model_dir(path)\n    texts = lora_texts(steps, seq)\n    r = memopro.finetune(path, texts, tokenizer=tok, budget=int(ceiling), device="cpu", seq_len=seq)\n    RESULT.update(\n        losses=r.losses[:steps],\n        loss_bits=[float(x).hex() for x in r.losses],\n        step_s=r.step_seconds,\n        ceiling=int(ceiling),\n    )\n\n\ndef csv_gen(data, gib, dims):\n    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\n    import csvgen\n\n    os.makedirs(data, exist_ok=True)\n    rows = int(float(gib) * (1 << 30) / csvgen.ROW_BYTES)\n    RESULT["fact_bytes"] = csvgen.write(\n        os.path.join(data, "fact.csv"), csvgen.fact_blocks(rows, int(dims))\n    )\n    RESULT["dim_bytes"] = csvgen.write(os.path.join(data, "dim.csv"), csvgen.dim_blocks(int(dims)))\n    RESULT.update(rows=rows, dims=int(dims))\n\n\ndef sort_mp(data, ceiling):\n    import memopro\n\n    h = hashlib.sha256()\n    st = memopro.sort(os.path.join(data, "fact.csv"), "id", out=h.update, budget=int(ceiling))\n    st.pop("runtime", None)\n    RESULT.update(hash=h.hexdigest(), stats=st, ceiling=int(ceiling))\n\n\ndef join_mp(data, ceiling):\n    import memopro\n\n    h = RowHash()\n    st = memopro.join(\n        os.path.join(data, "fact.csv"),\n        os.path.join(data, "dim.csv"),\n        "dim_id",\n        out=h.update,\n        budget=int(ceiling),\n    )\n    RESULT.update(hash=h.value(), stats=st, ceiling=int(ceiling))\n\n\ndef _duck(query, limit, sink, data):\n    """DuckDB with its memory limit; text columns joined by commas as the file\'s lines are."""\n    import duckdb\n    import pyarrow.compute as pc\n\n    tmp = os.path.join(data, "duck_tmp")\n    os.makedirs(tmp, exist_ok=True)\n    con = duckdb.connect(\n        config={\n            "memory_limit": f"{int(limit) >> 20}MB",\n            "temp_directory": tmp,\n            "threads": os.cpu_count(),\n        }\n    )\n    reader = con.execute(query).fetch_record_batch(1 << 18)\n    sink(b"header\\n")\n    for batch in reader:\n        line = pc.binary_join_element_wise(*batch.columns, ",")\n        sink(("\\n".join(line.to_pylist()) + "\\n").encode())\n    con.close()\n\n\ndef sort_duck(data, limit):\n    h = hashlib.sha256()\n    fact = os.path.join(data, "fact.csv").replace("\\\\", "/")\n    with open(os.path.join(data, "fact.csv"), "rb") as f:\n        header = f.readline()\n    first = [True]\n\n    def sink(b):\n        if first[0]:  # the file\'s own header instead of the placeholder\n            first[0] = False\n            h.update(header)\n            return\n        h.update(b)\n\n    _duck(\n        f"SELECT * FROM read_csv(\'{fact}\', all_varchar=true, header=true) "\n        "ORDER BY CAST(id AS BIGINT)",\n        limit,\n        sink,\n        data,\n    )\n    RESULT.update(hash=h.hexdigest(), memory_limit=int(limit))\n\n\ndef join_duck(data, limit):\n    h = RowHash()\n    fact = os.path.join(data, "fact.csv").replace("\\\\", "/")\n    dim = os.path.join(data, "dim.csv").replace("\\\\", "/")\n    _duck(\n        f"SELECT l.id, l.dim_id, l.a, l.b, l.text, r.name, r.w FROM read_csv(\'{fact}\', "\n        f"all_varchar=true, header=true) l JOIN read_csv(\'{dim}\', all_varchar=true, "\n        "header=true) r ON CAST(l.dim_id AS BIGINT) = CAST(r.dim_id AS BIGINT)",\n        limit,\n        h.update,\n        data,\n    )\n    RESULT.update(hash=h.value(), memory_limit=int(limit))\n\n\ndef sort_pandas(data):\n    import pandas as pd\n\n    t = time.time()\n    df = pd.read_csv(os.path.join(data, "fact.csv"), dtype=str, keep_default_na=False)\n    RESULT["read_s"] = time.time() - t\n    df = df.iloc[df["id"].astype("int64").argsort(kind="stable")]\n    h = hashlib.sha256()\n    for i in range(0, len(df), 1 << 20):  # the same text as the file\'s lines, piece by piece\n        h.update(\n            df.iloc[i : i + (1 << 20)]\n            .to_csv(index=False, header=i == 0, lineterminator="\\n")\n            .encode()\n        )\n    RESULT["hash"] = h.hexdigest()\n\n\ndef codec(mib):\n    import numpy as np\n\n    from memopro import rt\n\n    n = int(mib) << 20\n    rng = np.random.default_rng(0)\n    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\n    import csvgen\n\n    def csv_text():\n        text = bytearray()\n        for b in csvgen.fact_blocks(n // csvgen.ROW_BYTES + 1, 1_000_000):\n            text += b\n            if len(text) >= n:\n                break\n        return np.frombuffer(bytes(text[:n]), np.uint8)\n\n    x = np.arange(4096, dtype=np.float32) / 4096\n    kinds = {  # made one at a time\n        "CSV text": csv_text,\n        "int64 keys": lambda: rng.permutation(n // 8).astype(np.int64),\n        "uint16 image": lambda: (\n            np.resize((2000 + 900 * np.sin(np.arange(1 << 20) / 997.0)).astype(np.uint16), n // 2)\n            + rng.integers(0, 4, n // 2, dtype=np.uint16)\n        ),\n        "float32 field": lambda: np.resize(np.sin(9 * x)[None, :] * np.cos(7 * x)[:, None], n // 4),\n        "random bytes": lambda: rng.integers(0, 256, n, dtype=np.uint8),\n    }\n    out = []\n    for name, make in kinds.items():\n        raw = make().view(np.uint8).reshape(-1)\n        want = hashlib.sha256(raw).hexdigest()\n        r = rt.Runtime(budget=3 * n, process_budget=False, min_saving=0.0)\n        buf = r.alloc(raw.nbytes)\n        with buf.pin(write=True) as v:\n            np.frombuffer(v, np.uint8)[:] = raw\n        buf.evict()\n        s1 = r.stats()\n        with buf.pin() as v:\n            same = hashlib.sha256(np.frombuffer(v, np.uint8)).hexdigest() == want\n        s2 = r.stats()\n        c, d = s1["compress_seconds"], s2["decompress_seconds"]\n        out.append(\n            {\n                "kind": name,\n                "identical": same,\n                "ratio": raw.nbytes / s1["compress_out"] if s1["compress_out"] else None,\n                "compress_gbps": raw.nbytes / c / 1e9 if c else None,\n                "decompress_gbps": raw.nbytes / d / 1e9 if d else None,\n            }\n        )\n        del buf, r, raw\n    RESULT["codec"] = out\n\n\ndef env():\n    import platform\n    import subprocess\n\n    import psutil\n\n    RESULT.update(\n        platform=platform.platform(),\n        python=platform.python_version(),\n        processor=platform.processor(),\n        cpus=os.cpu_count(),\n        ram_bytes=psutil.virtual_memory().total,\n        swap_bytes=psutil.swap_memory().total,\n    )\n    r = subprocess.run(\n        [sys.executable, "-m", "memopro", "doctor", "--json"],\n        capture_output=True,\n        text=True,\n        check=False,\n    )\n    RESULT["doctor"] = r.stdout[-4000:] or r.stderr[-2000:]\n    import memopro\n\n    RESULT["memopro"] = memopro.__version__\n    for name in ("torch", "transformers", "peft", "duckdb", "pandas", "llama_cpp"):\n        try:\n            RESULT[name] = __import__(name).__version__\n        except Exception:  # noqa: BLE001 - recorded as missing\n            RESULT[name] = None\n\n\nKINDS = {\n    "env": env,\n    "gen_plain": gen_plain,\n    "gen_mp": gen_mp,\n    "gen_llama": gen_llama,\n    "lora_plain": lora_plain,\n    "lora_mp": lora_mp,\n    "csv_gen": csv_gen,\n    "sort_mp": sort_mp,\n    "join_mp": join_mp,\n    "sort_duck": sort_duck,\n    "join_duck": join_duck,\n    "sort_pandas": sort_pandas,\n    "codec": codec,\n}\n\nif __name__ == "__main__":\n    t0 = time.time()\n    try:\n        KINDS[KIND](*[int(a) if a.isdigit() else a for a in ARGS])\n        RESULT["status"] = "ok"\n    except MemoryError as e:\n        RESULT.update(status="oom", error=f"MemoryError: {e}"[:500])\n    except Exception as e:  # noqa: BLE001 - recorded\n        name = type(e).__name__\n        RESULT.update(\n            status="refused" if name == "BudgetExceeded" else "error",\n            error=f"{name}: {e}"[:1500],\n            trace=traceback.format_exc()[-3000:],\n        )\n    RESULT["wall_s"] = time.time() - t0\n    try:\n        RESULT.update(proc_stats())\n    except Exception as e:  # noqa: BLE001\n        RESULT["stats_error"] = str(e)\n    with open(OUT, "w", encoding="utf-8") as f:\n        json.dump(RESULT, f, default=str)\n', 'csvgen.py': '"""E069 (docs/research/0356) CSV data: deterministic fact and dimension tables.\n\nfact: ``id,dim_id,a,b,text`` with ``id`` a permutation of 0..rows-1 (unique, so a sort has one\nanswer), ``dim_id`` in 0..dims-1, ``text`` 10-30 words (~140-byte rows). dim: ``dim_id,name,w`` with\nevery dim_id once, in a shuffled order. Rows are made in blocks of ``BLOCK`` from seeds that depend\nonly on the block index, so any block can be made again on its own.\n\n    python -m experiments.e069_validation.csvgen fact.csv dim.csv --gib 30\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport sys\n\nimport numpy as np\n\nBLOCK = 1 << 20\nWORDS = np.array(\n    [\n        w.encode()\n        for w in ["alpha", "beta", "gamma", "delta", "epsilon", "zeta", "eta", "theta", "iota", "kappa", "lambda", "mu", "nu", "xi", "omicron", "pi", "rho", "sigma", "tau", "upsilon", "phi", "chi", "psi", "omega", "memory", "budget", "stream", "lossless", "ceiling", "buffer", "page", "chunk", "sort", "join", "merge", "key", "value", "table", "row", "column", "index", "file", "disk"]\n    ]\n)\nROW_BYTES = 140  # measured mean of a fact row (for --gib)\n\n\ndef _ids(rows: int) -> np.ndarray:\n    return np.random.default_rng(12345).permutation(rows).astype(np.int64)\n\n\ndef fact_blocks(rows: int, dims: int):\n    """Bytes of the fact table (header first), block by block."""\n    yield b"id,dim_id,a,b,text\\n"\n    ids = _ids(rows)\n    for start in range(0, rows, BLOCK):\n        n = min(BLOCK, rows - start)\n        rng = np.random.default_rng([7, start // BLOCK])\n        dim = rng.integers(0, dims, n)\n        a = rng.integers(0, 1_000_000, n)\n        b = rng.random(n)\n        lens = rng.integers(10, 31, n)\n        words = WORDS[rng.integers(0, len(WORDS), int(lens.sum()))]\n        lines, k = [], 0\n        for i in range(n):\n            m = int(lens[i])\n            text = b" ".join(words[k : k + m])\n            k += m\n            lines.append(b"%d,%d,%d,%.6f,%s" % (ids[start + i], dim[i], a[i], b[i], text))\n        yield b"\\n".join(lines) + b"\\n"\n\n\ndef dim_blocks(dims: int):\n    yield b"dim_id,name,w\\n"\n    order = np.random.default_rng(999).permutation(dims)\n    for start in range(0, dims, BLOCK):\n        n = min(BLOCK, dims - start)\n        rng = np.random.default_rng([11, start // BLOCK])\n        w = rng.random(n)\n        names = WORDS[rng.integers(0, len(WORDS), n)]\n        yield b"".join(\n            b"%d,%s%d,%.6f\\n" % (order[start + i], names[i], order[start + i] % 97, w[i])\n            for i in range(n)\n        )\n\n\ndef write(path: str, blocks) -> int:\n    total = 0\n    with open(path, "wb") as f:\n        for b in blocks:\n            f.write(b)\n            total += len(b)\n    return total\n\n\ndef main() -> None:\n    p = argparse.ArgumentParser()\n    p.add_argument("fact")\n    p.add_argument("dim")\n    p.add_argument("--gib", type=float, default=30.0)\n    p.add_argument("--dims", type=int, default=20_000_000)\n    a = p.parse_args()\n    rows = int(a.gib * (1 << 30) / ROW_BYTES)\n    n = write(a.fact, fact_blocks(rows, a.dims))\n    m = write(a.dim, dim_blocks(a.dims))\n    print(f"fact {rows} rows {n} bytes; dim {a.dims} rows {m} bytes", file=sys.stderr)\n\n\nif __name__ == "__main__":\n    main()\n'}
MIB, GIB = 1 << 20, 1 << 30


def load(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def save(path, rec):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(rec, f, default=str)


def left():
    return TIME_LIMIT_S - (time.time() - T_START)


def log(msg):
    line = f"[{time.time() - T_START:6.0f}s] {msg}"
    print(line, flush=True)
    if RUN:
        with open(os.path.join(RUN, "log.txt"), "a", encoding="utf-8") as f:
            f.write(line + "\n")


def pip(*args):
    return subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *args],
        capture_output=True,
        text=True,
        check=False,
    )


def setup():
    global RUN
    base = BASE or os.path.join(os.path.expanduser("~"), "memopro_e070")
    RUN = os.path.join(base, "runs", time.strftime("%Y%m%d-%H%M%S"))
    os.makedirs(os.path.join(RUN, "cases"), exist_ok=True)
    os.environ["HF_HOME"] = os.path.join(base, "hf")
    os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
    work = os.path.join(base, "work")
    os.makedirs(work, exist_ok=True)
    for name, src in WORKERS.items():
        with open(os.path.join(work, name), "w", encoding="utf-8") as f:
            f.write(src)
    log(f"results in {RUN}")
    r = pip("--pre", PIP_SPEC, "psutil", "duckdb", "pyarrow", "pandas")
    if r.returncode:
        raise RuntimeError(
            "installing memopro failed (a Windows wheel needs memopro 0.2.0b4 or "
            f"later):\n{r.stderr[-2000:]}"
        )
    r = pip(
        "llama-cpp-python",
        "--prefer-binary",
        "--extra-index-url",
        "https://abetlen.github.io/llama-cpp-python/whl/cpu",
    )
    log(
        "llama-cpp-python "
        + (
            "installed"
            if r.returncode == 0
            else f"not installed ({r.stderr[-300:]}); its cases are skipped"
        )
    )
    return base, work


def fetch(repo, pattern=None):
    from huggingface_hub import hf_hub_download, list_repo_files, snapshot_download

    t = time.time()
    if pattern is None:
        path = snapshot_download(
            repo, allow_patterns=["*.json", "*.safetensors", "*.txt", "tokenizer*", "merges.txt"]
        )
    else:
        names = [f for f in list_repo_files(repo) if pattern in f and f.endswith(".gguf")]
        if not names:
            raise FileNotFoundError(f"{repo}: no {pattern} GGUF")
        path = hf_hub_download(repo, min(names))
    log(f"{repo} {pattern or ''} ready ({time.time() - t:.0f}s)")
    return path


def case(key, expected_s, timeout, *args):
    """One worker run in a fresh process; the machine's pagefile and disk writes around it."""
    import psutil

    out = os.path.join(RUN, "cases", key + ".json")
    if expected_s > left():
        rec = {"case": key, "status": "skipped", "why": f"{left() / 60:.0f} min left"}
        save(out, rec)
        log(f"skip {key}: {rec['why']}")
        return rec
    swap0, disk0 = psutil.swap_memory().used, psutil.disk_io_counters().write_bytes
    t = time.time()
    log(f"run {key}")
    tmp = out + ".tmp"
    p = subprocess.Popen(
        [sys.executable, os.path.join(WORK, "worker_windows.py"), tmp, *[str(a) for a in args]],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=os.environ.copy(),
    )
    try:
        text, _ = p.communicate(timeout=min(timeout, max(60, left() + 300)))
    except subprocess.TimeoutExpired:
        p.kill()
        text, _ = p.communicate()
    rec = load(tmp) if os.path.exists(tmp) else {"status": "timeout" if p.returncode else "crashed"}
    if os.path.exists(tmp):
        os.remove(tmp)
    rec.update(
        case=key,
        wall_s=round(time.time() - t, 1),
        log_tail=(text or "")[-3000:],
        machine_swap_growth=psutil.swap_memory().used - swap0,
        machine_disk_written=psutil.disk_io_counters().write_bytes - disk0,
    )
    save(out, rec)
    log(f"  -> {rec.get('status')} ({rec['wall_s']}s)")
    return rec


def body():
    global WORK
    base, WORK = setup()
    case("env", 60, 300, "env")
    case("codec", 90, 600, "codec", 256)
    gen = fetch(GEN_MODEL)
    case("gen_plain", 240, 900, "gen_plain", gen, NEW_TOKENS)
    for c in GEN_CEILINGS:
        case(f"gen_mp_{c >> 20}", 360, 1200, "gen_mp", gen, NEW_TOKENS, c)
    lora = fetch(LORA_MODEL)
    case("lora_plain", 180, 900, "lora_plain", lora, LORA_STEPS, LORA_SEQ)
    for c in LORA_CEILINGS:
        case(f"lora_mp_{c >> 20}", 300, 1200, "lora_mp", lora, LORA_STEPS, LORA_SEQ, c)
    data = os.path.join(base, "data")
    if not os.path.exists(os.path.join(data, "dim.csv")):
        case("csv_gen", 150 * DATA_GIB / 4, 1800, "csv_gen", data, DATA_GIB, DATA_DIMS)
    case("sort_mp", 300, 1800, "sort_mp", data, DATA_CEILING)
    case("sort_duck", 240, 1800, "sort_duck", data, DATA_CEILING)
    case("join_mp", 240, 1800, "join_mp", data, DATA_CEILING)
    case("join_duck", 240, 1800, "join_duck", data, DATA_CEILING)
    if PANDAS:
        case("sort_pandas", 300, 900, "sort_pandas", data)
    try:
        import llama_cpp  # noqa: F401
    except ImportError:
        log("llama.cpp cases skipped (llama-cpp-python is not installed)")
    else:
        for kind in GGUF_KINDS:
            if left() > 240:
                case(f"gen_llama_{kind}", 180, 900, "gen_llama", fetch(GGUF_REPO, kind), NEW_TOKENS)
    summary()


def pandas_note(recs):
    """Whether pandas' sort gave the same bytes as memopro's (a second, independent check)."""
    p, m = recs.get("sort_pandas", {}), recs.get("sort_mp", {})
    if p.get("status") != "ok":
        return str(p.get("status"))
    return "same" if p.get("hash") == m.get("hash") else "different"


def summary():
    recs = {}
    for name in os.listdir(os.path.join(RUN, "cases")):
        r = load(os.path.join(RUN, "cases", name))
        recs[r["case"]] = r

    def ok(k):
        return recs.get(k, {}).get("status") == "ok"

    def mib(n):
        return "-" if n is None else f"{n / MIB:,.0f} MiB"

    rows = []

    def check(name, criterion, result, detail):
        rows.append(
            (
                name,
                criterion,
                "pass" if result is True else "fail" if result is False else "n/a",
                detail,
            )
        )

    plain = recs.get("gen_plain", {})
    for c in GEN_CEILINGS:
        r = recs.get(f"gen_mp_{c >> 20}", {})
        check(
            f"W1 gen {c >> 20} MiB",
            "text = the model loaded normally",
            ok("gen_plain") and ok(f"gen_mp_{c >> 20}") and r.get("texts") == plain.get("texts"),
            f"{r.get('status')}; {r.get('s_per_token', 0) or 0:.2f} vs "
            f"{plain.get('s_per_token', 0) or 0:.2f} s/token",
        )
    los = [recs.get(f"lora_mp_{c >> 20}", {}) for c in LORA_CEILINGS]
    check(
        "W2 lora",
        "completes at both ceilings, losses bit-identical",
        all(r.get("status") == "ok" for r in los)
        and los[0].get("loss_bits") == los[1].get("loss_bits"),
        str([r.get("status") for r in los]),
    )
    mine = [f"gen_mp_{c >> 20}" for c in GEN_CEILINGS]
    mine += [f"lora_mp_{c >> 20}" for c in LORA_CEILINGS] + ["sort_mp", "join_mp"]
    for k in mine:
        r = recs.get(k, {})
        ceiling = r.get("ceiling")
        if r.get("status") in ("ok", "refused"):
            check(
                f"W3 {k}",
                "peak working set <= ceiling",
                r.get("peak_bytes") is not None
                and ceiling is not None
                and r["peak_bytes"] <= ceiling,
                f"{mib(r.get('peak_bytes'))} vs {mib(ceiling)}",
            )
            check(
                f"W5 {k}",
                "written <= 64 MiB",
                (r.get("written_bytes") or 0) <= 64 * MIB,
                f"{mib(r.get('written_bytes'))}; machine pagefile "
                f"{mib(r.get('machine_swap_growth'))}",
            )
    check(
        "W4 sort",
        "memopro.sort SHA-256 = DuckDB's",
        ok("sort_mp") and ok("sort_duck") and recs["sort_mp"]["hash"] == recs["sort_duck"]["hash"],
        f"{recs.get('sort_mp', {}).get('hash', '-')[:16]} / "
        f"{recs.get('sort_duck', {}).get('hash', '-')[:16]}; pandas: {pandas_note(recs)}",
    )
    check(
        "W4 join",
        "memopro.join rows = DuckDB's",
        ok("join_mp") and ok("join_duck") and recs["join_mp"]["hash"] == recs["join_duck"]["hash"],
        f"{recs.get('join_mp', {}).get('hash')} / {recs.get('join_duck', {}).get('hash')}",
    )
    lines = [
        "# E070: memopro on a Windows PC (CPU), local runtime",
        "",
        f"{recs.get('env', {}).get('platform')}, {recs.get('env', {}).get('processor')}, "
        f"{mib(recs.get('env', {}).get('ram_bytes'))} RAM, memopro "
        f"{recs.get('env', {}).get('memopro')}; elapsed {(time.time() - T_START) / 60:.0f} min",
        "",
        "| check | criterion | result | detail |",
        "|---|---|---|---|",
    ]
    lines += [f"| {a} | {b} | {c} | {d} |" for a, b, c, d in rows]
    lines += [
        "",
        "## All cases (report)",
        "",
        "| case | status | wall s | s/token or s/step | peak | written | pagefile growth |",
        "|---|---|---|---|---|---|---|",
    ]
    for k in sorted(recs):
        r = recs[k]
        speed = r.get("s_per_token") or (
            sum(r["step_s"][1:]) / max(1, len(r["step_s"]) - 1) if r.get("step_s") else None
        )
        lines.append(
            f"| {k} | {r.get('status')} | {r.get('wall_s')} | "
            f"{'-' if speed is None else f'{speed:.3f}'} | {mib(r.get('peak_bytes'))} | "
            f"{mib(r.get('written_bytes'))} | {mib(r.get('machine_swap_growth'))} |"
        )
    for r in recs.get("codec", {}).get("codec", []):
        lines.append(
            f"- codec {r['kind']}: ratio {r['ratio'] and round(r['ratio'], 2)}, "
            f"{r['compress_gbps'] and round(r['compress_gbps'], 2)} / "
            f"{r['decompress_gbps'] and round(r['decompress_gbps'], 2)} GB/s, "
            f"identical {r['identical']}"
        )
    bad = [(k, r) for k, r in sorted(recs.items()) if r.get("status") not in ("ok", None)]
    if bad:
        lines += ["", "## Not ok", ""]
        lines += [
            f"- `{k}` ({r.get('status')}): {str(r.get('error') or r.get('why'))[:300]}"
            for k, r in bad
        ]
    text = "\n".join(lines) + "\n"
    with open(os.path.join(RUN, "summary.md"), "w", encoding="utf-8") as f:
        f.write(text)
    archive = shutil.make_archive(RUN, "zip", RUN)
    print(text)
    print(f"send this file: {archive}")


RUN = None
body()
